# Aula 07 - Notebook: Validade de Argumentos na Máquina de Envasamento

Neste notebook implementamos a classe **`ProvadorDedutivoFormal`** (Motor de Inferência SAT Solver e Tabela-Verdade) para validar as regras do SCADA no envasamento de copos plásticos. O autograder avaliará a consistência dos intertravamentos para os Setores 000 a 500.

In [1]:
import itertools
from typing import List, Dict, Callable, Any

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

class ProvadorDedutivoFormal:
    @staticmethod
    def verificar_argumento_tabela_verdade(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Verifica a validade do argumento por tabela-verdade exaustiva (2^n estados).
        """
        n = len(variaveis)
        total_estados = 2 ** n
        linhas_criticas = 0
        linhas_validas = 0
        contraexemplos = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas):
                linhas_criticas += 1
                if conclusao(env):
                    linhas_validas += 1
                else:
                    contraexemplos.append(env)
                    
        valido = (linhas_criticas > 0) and (linhas_criticas == linhas_validas)
        return {
            "Total Estados (2^n)": total_estados,
            "Válido": valido,
            "Resultado Semântico": "ARGUMENTO VÁLIDO" if valido else "FALÁCIA / INVÁLIDO",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def verificar_por_refutacao(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        SAT Solver Approach: Reductio ad Absurdum.
        """
        n = len(variaveis)
        modelos_refutacao = []
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos_refutacao.append(env)
                
        is_insatisfativel = len(modelos_refutacao) == 0
        return {
            "Refutação Bem-Sucedida": is_insatisfativel,
            "Conclusão": "VÁLIDO (Contradição)" if is_insatisfativel else "INVÁLIDO (Falhou)"
        }

print("[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!")


[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!


In [2]:
# ==============================================================================
# BATERIA DE TESTES - AUTOGRADER: MÁQUINA DE ENVASAMENTO DE COPOS
# ==============================================================================

# 1. Modus Ponens (MP): e1 -> parada, e1 |- parada
vars_mp = ['e1', 'parada']
res_mp = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_mp, 
    [lambda e: (not e['e1']) or e['parada'], lambda e: e['e1']], 
    lambda e: e['parada']
)

# 2. Modus Tollens (MT): v7_vacuo -> p1_pressostato, not p1_pressostato |- not v7_vacuo
vars_mt = ['v7_vacuo', 'p1_pressostato']
res_mt = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_mt, 
    [lambda e: (not e['v7_vacuo']) or e['p1_pressostato'], lambda e: not e['p1_pressostato']], 
    lambda e: not e['v7_vacuo']
)

# 3. Silogismo Hipotético (SH): sem_copo -> bloqueia_dosador, bloqueia_dosador -> bloqueia_bico |- sem_copo -> bloqueia_bico
vars_sh = ['sem_copo', 'bloqueia_dosador', 'bloqueia_bico']
res_sh = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_sh, 
    [lambda e: (not e['sem_copo']) or e['bloqueia_dosador'], lambda e: (not e['bloqueia_dosador']) or e['bloqueia_bico']], 
    lambda e: (not e['sem_copo']) or e['bloqueia_bico']
)

# 4. Silogismo Disjuntivo (SD): t1 or bloqueia_bico, not t1 |- bloqueia_bico
vars_sd = ['t1', 'bloqueia_bico']
res_sd = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_sd,
    [lambda e: e['t1'] or e['bloqueia_bico'], lambda e: not e['t1']],
    lambda e: e['bloqueia_bico']
)

# 5. Resolução Proposicional: (sem_copo or parada), (not sem_copo or bloqueia_dosador) |- (parada or bloqueia_dosador)
vars_res = ['sem_copo', 'bloqueia_dosador', 'parada']
res_res = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_res, 
    [lambda e: e['sem_copo'] or e['parada'], lambda e: (not e['sem_copo']) or e['bloqueia_dosador']], 
    lambda e: e['parada'] or e['bloqueia_dosador']
)

# 6. Dilema Construtivo (DC): (e1 -> parada) and (sem_copo -> bloqueia_dosador), e1 or sem_copo |- parada or bloqueia_dosador
vars_dc = ['e1', 'parada', 'sem_copo', 'bloqueia_dosador']
res_dc = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_dc,
    [lambda e: ((not e['e1']) or e['parada']) and ((not e['sem_copo']) or e['bloqueia_dosador']), lambda e: e['e1'] or e['sem_copo']],
    lambda e: e['parada'] or e['bloqueia_dosador']
)

# 7. Falácia da Afirmação do Consequente (INVÁLIDO): e1 -> parada, parada |- e1
vars_fal = ['e1', 'parada']
res_fal = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_fal, 
    [lambda e: (not e['e1']) or e['parada'], lambda e: e['parada']], 
    lambda e: e['e1']
)

relatorio = [
    {"Esquema": "Modus Ponens (MP)", "Variáveis": "e1, parada", "Válido": res_mp["Válido"], "Status": res_mp["Resultado Semântico"]},
    {"Esquema": "Modus Tollens (MT)", "Variáveis": "v7_vacuo, p1_press", "Válido": res_mt["Válido"], "Status": res_mt["Resultado Semântico"]},
    {"Esquema": "Silogismo Hipot. (SH)", "Variáveis": "sem_copo, bloq_dos, bloq_bico", "Válido": res_sh["Válido"], "Status": res_sh["Resultado Semântico"]},
    {"Esquema": "Silogismo Disj. (SD)", "Variáveis": "t1, bloqueia_bico", "Válido": res_sd["Válido"], "Status": res_sd["Resultado Semântico"]},
    {"Esquema": "Resolução", "Variáveis": "sem_copo, bloq_dos, parada", "Válido": res_res["Válido"], "Status": res_res["Resultado Semântico"]},
    {"Esquema": "Dilema Construtivo", "Variáveis": "e1, parada, sem_copo, bloq_dos", "Válido": res_dc["Válido"], "Status": res_dc["Resultado Semântico"]},
    {"Esquema": "Afirmação Consequente", "Variáveis": "e1, parada", "Válido": res_fal["Válido"], "Status": res_fal["Resultado Semântico"]}
]

print("=== RELATÓRIO DO AUTOGRADER: LÓGICA DO SCADA DA ENVASADORA ===")
print(formatar_tabela(relatorio))

assert res_mp["Válido"] is True, "Falha no MP"
assert res_mt["Válido"] is True, "Falha no MT"
assert res_sh["Válido"] is True, "Falha no SH"
assert res_sd["Válido"] is True, "Falha no SD"
assert res_res["Válido"] is True, "Falha na Resolução"
assert res_dc["Válido"] is True, "Falha no Dilema Construtivo"
assert res_fal["Válido"] is False, "Falha na detecção de Falácia"

print("\n[OK] Autograder validou todos os 2^n estados do sistema com sucesso!")


=== RELATÓRIO DO AUTOGRADER: LÓGICA DO SCADA DA ENVASADORA ===
Esquema                 | Variáveis                          | Válido | Status            
------------------------+------------------------------------+--------+-------------------
Modus Ponens (MP)       | e1, parada                         | True   | ARGUMENTO VÁLIDO  
Modus Tollens (MT)      | v7_vacuo, p1_press                 | True   | ARGUMENTO VÁLIDO  
Silogismo Hipot. (SH)   | sem_copo, bloq_dos, bloq_bico      | True   | ARGUMENTO VÁLIDO  
Silogismo Disj. (SD)    | t1, bloqueia_bico                  | True   | ARGUMENTO VÁLIDO  
Resolução               | sem_copo, bloq_dos, parada         | True   | ARGUMENTO VÁLIDO  
Dilema Construtivo      | e1, parada, sem_copo, bloq_dos     | True   | ARGUMENTO VÁLIDO  
Afirmação Consequente   | e1, parada                         | False  | FALÁCIA / INVÁLIDO

[OK] Autograder validou todos os 2^n estados do sistema com sucesso!
